<a href="https://colab.research.google.com/github/NicolasHillingdale/Part1-Basic_Agent_Part_1_Colab.ipynb/blob/test/Part1/Basic_Agent_Part_1_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Last Updated: August 2026
(Tested on Google Colab)

In [19]:
!python --version

Python 3.12.13


In [30]:
!pip install -q -U "tavily-python>=0.7.0" "crewai[tools]" "python-dotenv" "langchain-anthropic" "pandas" "langchain_community"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.5/60.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.9/753.9 kB 38.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.1.20 requires langchain-community<0.1,>=0.0.38, but you have langchain-community 0.4.2 which is incompatible.
langchain 0.1.20 requires langchain-core<0.2.0,>=0.1.52, but you have langchain-core 1.6.1 which is incompatible.
langchain 0.1.20 requires langchain-

In [ ]:
# Restart code
import os
os.kill(os.getpid(), 9)

In [6]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [7]:
import crewai
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from tavily import TavilyClient
import json

In [11]:
llm = LLM(model="anthropic/claude-sonnet-4-5-20250929", temperature=0.2)
client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@tool("Tavily Search")
def tavily_search(query: str) -> str:
  """Search the web for recent renewable energy deal news. Returns JSON results with URLs."""
  r = client.search(
        query=query,
        search_depth="advanced",
        topic="news",
        time_range="week",
        max_results=10,
        include_raw_content=True,
    )
  return json.dumps(r["results"], indent=2)


@tool("Tavily Extract")
def tavily_extract(urls: str) -> str:
    """Extract full page content from one or more comma-separated URLs."""
    return json.dumps(client.extract([u.strip() for u in urls.split(",")]), indent=2)

In [10]:
from pydantic import BaseModel, Field
from typing import List, Optional

class Opportunity(BaseModel):
  company_name: str
  country: str
  region: str
  technology: str
  deal_type: str
  stage: str
  signal: str
  deal_size: Optional[str] = None
  source_url: str
  date_announced: Optional[str] = None

class OpportunityList(BaseModel):
  opportunities: List[Opportunity]

class ScoredOpportunity(Opportunity):
  mandate_fit_score: int = Field(ge=1, le=10)
  financing_need_score: int = Field(ge=1, le=10)
  engagement_likelihood_score: int = Field(ge=1, le=10)
  overall_score: float
  rationale: str
  suggested_role: str              # Advisor / Equity / Debt / Monitor

class ScoredList(BaseModel):
  scored: List[ScoredOpportunity]

In [12]:
researcher = Agent(
    role="Renewable Energy Origination Researcher",
    goal=(
        "Find companies, IPPs and project developers in the African and European "
        "renewable energy markets that show evidence of needing project or growth finance."
    ),
    backstory=(
        "You are a research analyst at Hillingdale Capital, an independent capital-raising "
        "and corporate finance advisory firm with offices in Cape Town and Paris. You track "
        "M&A of developers/IPPs/portfolios, project financing and financial close announcements, "
        "equity raises, new utility-scale project announcements, asset sales, and fund closes "
        "targeting African or European renewables — across solar, wind, storage, hydro, hybrid "
        "and green hydrogen, at any stage from development to operational refinancing. "
        "Focus countries: South Africa, Kenya, Nigeria, Egypt, Morocco, Ghana, Senegal, "
        "UK, France, Germany, Spain, Italy. You favour reputable trade and financial press "
        "and you never report a company without a verifiable source URL."
    ),
    tools=[tavily_search, tavily_extract],
    llm=llm,
    allow_delegation=False,
    max_iter=10,
    verbose=True
)

In [13]:
evaluator = Agent(
    role="Origination Evaluator",
    goal=(
        "Score each researched opportunity against the Hillingdale mandate and judge how "
        "likely it is to need financing and to engage with an independent advisor."
    ),
    backstory=(
        "You are an evaluation analyst at Hillingdale Capital. You are deliberately sceptical: "
        "you discount vague announcements, already-funded projects, and companies large enough "
        "to have in-house capital markets teams or existing tier-1 advisors. You score on three "
        "axes, 1-10 each:\n"
        "  mandate_fit — does this sit inside Hillingdale's sector, geography and instrument set?\n"
        "  financing_need — how strong is the evidence they need capital now (1 = fully funded, "
        "10 = explicitly in-market seeking debt or equity)?\n"
        "  engagement_likelihood — how plausible is it they would work with an independent "
        "advisor (1 = mega-cap with a house bank, 10 = mid-size developer with no named advisor)?\n"
        "overall_score is the mean of the three, rounded to one decimal."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

In [14]:
research = Task(
    description=(
        "Search the web for renewable energy deal activity and capital-raising signals in "
        "Africa and Europe announced in the last 14 days. Run several distinct searches across "
        "deal types, technologies and the focus countries. Open the most promising articles to "
        "confirm details. Return only companies with a real, verifiable source URL — never invent "
        "a company, a figure or a link. Aim for 10-15 solid entries rather than a long weak list."
    ),
    expected_output="A structured list of opportunities matching the OpportunityList schema.",
    output_pydantic=OpportunityList,
    agent=researcher,
)

evaluate = Task(
    description=(
        "Score every opportunity from the research task using the three-axis rubric in your "
        "backstory. Give a one-to-two sentence rationale per company and assign a suggested "
        "role for Hillingdale: Advisor, Equity, Debt, or Monitor. Do not add companies that "
        "were not in the research output."
    ),
    expected_output="A structured list matching the ScoredList schema.",
    output_pydantic=ScoredList,
    context=[research],          # <-- this is the handoff, not {companies_list}
    agent=evaluator,
)

In [16]:
crewman = Crew(
    agents=[researcher, evaluator],
    tasks=[research, evaluate],
    process=Process.sequential,
    verbose=True,
)

result = await crewman.kickoff_async()     # <-- the fix

Streaming output truncated to the last 5000 lines.
│  More](http://www.energychoicematters.com/stories/20240319b.html)\n\n\u00a0  │
│  \n\n[PUC Approves Utility's New Purchase Of Receivables Program In Nisi     │
│  Order](http://www.energychoicematters.com/stories/20240319a.html)\n\nA PUC  │
│  approved a settlement agreement between...\n\n[Click for                    │
│  More](http://www.energychoicematters.com/stories/20240319a.html)\n\n\u00a0  │
│  \n\n[Google, Microsoft, Nucor To Aggregate Demand To Support Advanced       │
│  Clean Electricity                                                           │
│  Technologies](http://www.energychoicematters.com/stories/20240319c.html)\n  │
│  \nGoogle LLC, Microsoft Corporation, and Nucor Corporation today announced  │
│  they will ...\n\n[Click for                                                 │
│  More](http://www.energychoicematters.com/stories/20240319c.html)\n\n\u00a0  │
│  \n\n[Arcadia Names New                                 

ERROR:root:Anthropic API call failed: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'prompt is too long: 639462 tokens > 200000 maximum'}, 'request_id': 'req_011CearvEdWJu6tDTnKRSyXa'}


An unknown error occurred. Please check the details below.╭──────────────────────────────── ❌ LLM Error ────────────────────────────────╮
│                                                                              │
│  LLM Call Failed                                                             │
│  Error: Anthropic API call failed: Error code: 400 - {'type': 'error',       │
│  'error': {'type': 'invalid_request_error', 'message': 'prompt is too long:  │
│  639462 tokens > 200000 maximum'}, 'request_id':                             │
│  'req_011CearvEdWJu6tDTnKRSyXa'}                                             │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

Error details: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'prompt is too long: 639462 tokens > 200000 maximum'}, 'request_id': 'req_011CearvEdWJu6tDTnKRSyXa'}



ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'prompt is too long: 639462 tokens > 200000 maximum'}, 'request_id': 'req_011CearvEdWJu6tDTnKRSyXa'}


Streaming output truncated to the last 5000 lines.
│  national strategy that places energy security at the centre of Zambia's...  │
│  Read more on                                                                │
│  [https://lnkd.in/dwz5eAN3](https://lnkd.in/dwz5eAN3?trk=public_post-text)\  │
│  n\n![No alternative text description for this                               │
│  image]()\n\n\n![]()\n![]()\n\n\n\n\n\n\n\n\n\n\nTo view or add a comment,   │
│  [sign                                                                       │
│  in](https://www.linkedin.com/signup/cold-join?session_redirect=https%3A%2F  │
│  %2Fwww.linkedin.com%2Fposts%2Fbarnabas-zulu-266099171_zambias-energy-revol  │
│  ution-how-massive-power-activity-7488853390056267777-M4wv&trk=public_post_  │
│  feed-cta-banner-cta)\n\n![View organization page for New Energy Industry    │
│  Association for Asia and the Pacific]()\n\n279 followers\n\nThe new 15th    │
│  Five-Year Plan for Renewable Energy (2026\u20132030), j

...

ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'prompt is too long: 681595 tokens > 200000 maximum'}, 'request_id': 'req_011CearwpFrbfmPGCMCypqw2'}


Streaming output truncated to the last 5000 lines.
│  innovative start-up in the area of developing power conversion/wireless     │
│  power solutions and acting as a driver of innovation in the power           │
│  electronics sector.\n\nThe project entails positive knowledge spill-overs   │
│  related to the development of energy efficient power conversion and         │
│  wireless power applications in millions of devices, which enables           │
│  significant reductions in power consumption and related greenhouse gas      │
│  emissions, and hence strengthens the EU's innovative position in power      │
│  electronics markets.\n\nEIB's financial contribution facilitates the SME's  │
│  sustainable growth and helps to maintain its technological edge, despite    │
│  the adverse economic conditions resulting from the COVID-19 outbreak.       │
│  Furthermore, it provides a positive signalling effect to crowd-in further   │
│  financing from private investors.\n\nEnvironmental     

ERROR:root:Anthropic API call failed: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'prompt is too long: 2696723 tokens > 200000 maximum'}, 'request_id': 'req_011Ceary3hmJ5ibXE3K1N5Cb'}
ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'prompt is too long: 2696723 tokens > 200000 maximum'}, 'request_id': 'req_011Ceary3hmJ5ibXE3K1N5Cb'}


Streaming output truncated to the last 5000 lines.
│  ost-text) [Isaac Kwaku                                                      │
│  F.](https://ae.linkedin.com/in/isaackwakufokuojr?trk=public_post-text)      │
│  [Erika                                                                      │
│  Amoako-Agyei](https://www.linkedin.com/in/erika-amoako-agyei-4a68824?trk=p  │
│  ublic_post-text)\n[#empowerafrica](https://www.linkedin.com/signup/cold-jo  │
│  in?session_redirect=https%3A%2F%2Fwww.linkedin.com%2Ffeed%2Fhashtag%2Fempo  │
│  werafrica&trk=public_post-text)                                             │
│  [#africa](https://www.linkedin.com/signup/cold-join?session_redirect=https  │
│  %3A%2F%2Fwww.linkedin.com%2Ffeed%2Fhashtag%2Fafrica&trk=public_post-text)   │
│  [#investinafrica](https://www.linkedin.com/signup/cold-join?session_redire  │
│  ct=https%3A%2F%2Fwww.linkedin.com%2Ffeed%2Fhashtag%2Finvestinafrica&trk=pu  │
│  blic_post-text)                                        

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'prompt is too long: 2696723 tokens > 200000 maximum'}, 'request_id': 'req_011Ceary3hmJ5ibXE3K1N5Cb'}

In [ ]:
import pandas as pd

df = pd.DataFrame([s.model_dump() for s in result.pydantic.scored])
df = df.sort_values("overall_score", ascending=False)
df.to_csv("hillingdale_opportunities.csv", index=False)
df.head(20)